In [ ]:
%run ./_common

In [ ]:
#%run ./Classroom-Setup-0

In [ ]:
class Token:
    def __init__(self, config_path='./../var/credentials.cfg'):
        self.config_path = config_path
        self.token = self.get_credentials()

    def get_credentials(self):
        import configparser
        import os

        # Check if the file exists
        if not os.path.exists(self.config_path):
            # No saved PAT (e.g. "0 - Generate Tokens" was skipped, or running as a job):
            # use the token of the current notebook session instead.
            print(f"No credentials file at {self.config_path}; using the notebook session token.")
            return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

        config = configparser.ConfigParser()
        config.read(self.config_path)

        if 'DEFAULT' not in config:
            raise KeyError("The section 'DEFAULT' was not found in the configuration file.")

        if 'db_token' not in config['DEFAULT']:
            raise KeyError("The key 'db_token' was not found in the 'DEFAULT' section of the configuration file.")

        token = config['DEFAULT']['db_token']
        
        # Never print the full token; show only enough to confirm which one was loaded
        print(f"db_token loaded: {token[:4]}...{token[-4:]}")

        return token

In [ ]:
# Use the Token class to get the token
token_obj = Token()
token = token_obj.token

In [ ]:
def create_features_table(self):
    from pyspark.sql.functions import col

    # Define active catalog and schema
    spark.sql(f"USE CATALOG {SA.catalog_name}")
    spark.sql(f"USE {SA.schema_name}")

    # Read the dataset
    dataset_path = f"{SA.paths.datasets.banking}/banking/loan-clean.csv"
    loan_df = spark.read.csv(dataset_path, header="true", inferSchema="true", multiLine="true", escape='"')

    # Select columns of interest and replace spaces with underscores
    loan_df = loan_df.selectExpr(
        "ID",
        "Age",
        "Experience",
        "Income",
        "`ZIP Code` as ZIP_Code",
        "Family",
        "CCAvg",
        "Education",
        "Mortgage",
        "`Personal Loan` as Personal_Loan",
        "`Securities Account` as Securities_Account",
        "`CD Account` as CD_Account",
        "Online",
        "CreditCard"
    )

    # Save df as delta table using Delta API
    loan_df.write.format("delta").mode("overwrite").saveAsTable("bank_loan")

# Monkey patch the method into SWSAcademyHelper
SWSAcademyHelper.add_method(create_features_table)

In [ ]:
SA = SWSAcademyHelper()  # Create the SA object
SA.init()                                           # Performs basic intialization including creating schemas and catalogs
SA.create_features_table()